
# 20.21 — Cold-start vs continuation — v2 autônomo
## Controle causal da memória do otimizador na fronteira clássica

### Correção desta versão

A primeira versão dependia de um `SCENARIO_RUNNER` existente no kernel. Isso foi inadequado para um notebook aberto separadamente da 20.20.

Esta versão **não depende mais de `SCENARIO_RUNNER`**. Ela reconstrói diretamente o mesmo problema físico usado na 20.20 a partir de:

1. `merge.pkl` — para recuperar \(\mu\) e \(\Sigma\);
2. `fine_boundary_quantum_scan.csv` — para recuperar os pontos e os resultados forward/backward já calculados;
3. o mesmo ansatz de Dicke rastreável e o mesmo objetivo de energia da 20.20.

O teste central é:

\[
\text{forward continuation}
\quad\text{vs}\quad
\text{backward continuation}
\quad\text{vs}\quad
N_{\rm cold}\text{ inicializações independentes}.
\]

### Importante

Cada réplica cold-start começa de um \(\theta_0\) independente amostrado no domínio angular dos blocos.  
Não é usado `ACTION_THETA`, não é usado o ótimo clássico como inicialização e não é usado warm start.

O ótimo clássico continua sendo usado **somente como label de avaliação** de \(P_{\rm opt}\) e do gap de energia.


In [ ]:

from __future__ import annotations

from itertools import combinations
from pathlib import Path
import ast
import hashlib
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from IPython.display import display

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

# ============================================================
# CONFIGURAÇÃO
# ============================================================

RANDOM_SEED = 42
Q_VALUE = 0.5
RISK_FREE = 0.0475
ENERGY_ATOL = 1e-8

N_COLD_STARTS = 10
COLD_MAXITER = 420

P_OPT_SUCCESS = 0.90
P_OPT_STRONG = 0.99
ENERGY_GAP_TOL = 1e-6

QGT_FD_STEP = 1e-4
QGT_REL_NULL_TOL = 1e-6
ACTIVE_TRACE_COVERAGE = 0.95

ROOT = Path.cwd()
OUT_ROOT = ROOT / "vqe_r" / "validation_v20_21_cold_start"
TABLE_DIR = OUT_ROOT / "tables"
FIGURE_DIR = OUT_ROOT / "figures"
CHECKPOINT_DIR = OUT_ROOT / "checkpoints"

for d in [OUT_ROOT, TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT.resolve())
print("Saídas 20.21 =", OUT_ROOT.resolve())
print("N cold starts por ponto =", N_COLD_STARTS)
print("COBYLA maxiter por cold start =", COLD_MAXITER)



## Célula 3 — localizar e reconstruir o Hamiltoniano original

A 20.20 usa

\[
E(x)=q\,x^\top\Sigma x-(1-q)\mu^\top x+r_f.
\]

Aqui o `merge.pkl` é lido apenas para recuperar o mesmo \(\mu\) e a mesma \(\Sigma\).


In [ ]:

# ============================================================
# LOCALIZAR merge.pkl
# ============================================================

def find_merge_pkl():
    explicit = globals().get("MERGE_PKL", None)
    candidates = []
    if explicit is not None:
        candidates.append(Path(explicit))

    candidates += [
        ROOT / "merge.pkl",
        Path.home() / "Downloads" / "merge.pkl",
        Path.home() / "Desktop" / "merge.pkl",
    ]

    for p in candidates:
        try:
            if p.expanduser().is_file():
                return p.expanduser().resolve()
        except Exception:
            pass

    # Busca curta somente em diretórios plausíveis.
    for base in [ROOT, Path.home() / "Downloads"]:
        try:
            if base.exists():
                hits = list(base.rglob("merge.pkl"))
                if hits:
                    return hits[0].resolve()
        except Exception:
            pass

    raise FileNotFoundError(
        "merge.pkl não encontrado. Coloque-o no mesmo diretório do notebook "
        "ou em ~/Downloads, ou defina MERGE_PKL = Path(r'...')."
    )

MERGE_PKL = find_merge_pkl()
print("merge.pkl =", MERGE_PKL)

loaded = pd.read_pickle(MERGE_PKL)
if isinstance(loaded, pd.DataFrame):
    merge_df = loaded.copy()
elif isinstance(loaded, list):
    merge_df = pd.DataFrame(loaded)
elif isinstance(loaded, dict):
    merge_df = pd.DataFrame(loaded)
else:
    raise TypeError(f"Tipo não suportado em merge.pkl: {type(loaded)}")

if merge_df.empty:
    raise ValueError("merge.pkl vazio.")

COLUMN_ALIASES = {
    "tickers": ["tickers", "assets", "asset_names"],
    "assets_return": ["assets_return", "assets_returns", "expected_returns", "mu"],
    "covariance": ["covariance", "covariance_matrix", "sigma"],
    "best_parameters": ["best_parameters", "theta", "theta_final"],
}

def first_existing_column(frame, aliases, required=True):
    for name in aliases:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(f"Nenhuma coluna encontrada entre: {aliases}")
    return None

RESOLVED_COLUMNS = {
    key: first_existing_column(merge_df, aliases)
    for key, aliases in COLUMN_ALIASES.items()
}

def parse_serialized(value):
    if isinstance(value, (np.ndarray, list, tuple, dict, pd.Series, pd.Index, pd.DataFrame)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                pass
        cleaned = text.strip("[]()")
        arr = np.fromstring(cleaned.replace(",", " "), sep=" ")
        if arr.size:
            return arr
    return value

def parse_tickers(value):
    parsed = parse_serialized(value)
    if isinstance(parsed, str):
        items = [x.strip() for x in parsed.replace(";", ",").split(",")]
    elif isinstance(parsed, dict):
        items = list(parsed.keys())
    else:
        items = list(parsed)
    tickers = [str(x).strip().strip("'\"") for x in items]
    if not tickers or any(not x for x in tickers):
        raise ValueError(f"Tickers inválidos: {value}")
    return tickers

def parse_vector(value, tickers=None):
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.Series):
        if tickers is not None and set(tickers).issubset(set(parsed.index.astype(str))):
            return parsed.reindex(tickers).to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        if tickers is not None and set(tickers).issubset(set(map(str, parsed.keys()))):
            return np.asarray([parsed[t] for t in tickers], dtype=float)
        return np.asarray(list(parsed.values()), dtype=float)
    return np.asarray(parsed, dtype=float).reshape(-1)

def parse_matrix(value, tickers=None):
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.DataFrame):
        if tickers is not None:
            return parsed.loc[tickers, tickers].to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        frame = pd.DataFrame(parsed)
        if (
            tickers is not None
            and set(tickers).issubset(frame.index)
            and set(tickers).issubset(frame.columns)
        ):
            return frame.loc[tickers, tickers].to_numpy(dtype=float)
        return frame.to_numpy(dtype=float)

    arr = np.asarray(parsed, dtype=float)
    if arr.ndim == 1:
        n = int(round(np.sqrt(arr.size)))
        if n * n != arr.size:
            raise ValueError("Covariância 1D não forma matriz quadrada.")
        arr = arr.reshape(n, n)
    return arr

valid = merge_df[RESOLVED_COLUMNS["best_parameters"]].notna()
if not bool(valid.any()):
    raise ValueError("Nenhuma linha com best_parameters/theta válida no merge.pkl.")

first_row = merge_df.loc[valid.idxmax()]
tickers = parse_tickers(first_row[RESOLVED_COLUMNS["tickers"]])
mu = parse_vector(first_row[RESOLVED_COLUMNS["assets_return"]], tickers=tickers)
sigma = parse_matrix(first_row[RESOLVED_COLUMNS["covariance"]], tickers=tickers)
sigma = 0.5 * (sigma + sigma.T)

N_ASSETS = len(tickers)
if mu.shape != (N_ASSETS,):
    raise ValueError(f"mu tem shape {mu.shape}; esperado {(N_ASSETS,)}")
if sigma.shape != (N_ASSETS, N_ASSETS):
    raise ValueError(f"sigma tem shape {sigma.shape}; esperado {(N_ASSETS, N_ASSETS)}")
if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("mu/sigma contêm valores não finitos.")

mu_scale = float(np.std(mu))
if mu_scale <= 0:
    mu_scale = max(float(np.max(np.abs(mu))), 1.0)

print("n_assets =", N_ASSETS)
print("tickers =", tickers)
print("mu_scale =", mu_scale)
print("min eig(sigma) =", float(np.linalg.eigvalsh(sigma).min()))



## Célula 4 — mesmo ansatz de Dicke rastreável da 20.20

O número de parâmetros é reconstruído para cada cardinalidade:

\[
N_\theta(k)=\frac{k(2n-k-1)}{2}.
\]

Para \(n=10\):
- \(k=3\Rightarrow N_\theta=24\)
- \(k=4\Rightarrow N_\theta=30\)
- \(k=5\Rightarrow N_\theta=35\)


In [ ]:

# ============================================================
# ANSATZ DE DICKE RASTREÁVEL
# ============================================================

def CY_parameterized(identifier):
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")

def CCY_parameterized(identifier):
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")

def dicke_parameter_count(n_value, k_value):
    return int(k_value * (2 * n_value - k_value - 1) / 2)

def build_tracked_dicke_ansatz(n_value, k_value, seed):
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)

        initial_x_qubits = []
        for excitation_index in range(k_value):
            qubit = n_value - excitation_index - 1
            qc.x(qubit)
            initial_x_qubits.append(qubit)

        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"

                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    decomposed = qc.decompose()
    ordered_parameters = list(decomposed.parameters)
    parameter_to_index = {p: i for i, p in enumerate(ordered_parameters)}

    rows = []
    for record in records:
        rec = record.copy()
        parameter_object = rec.pop("parameter_object")
        rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **rec,
        })

    structure_df = (
        pd.DataFrame(rows)
        .sort_values("theta_index")
        .reset_index(drop=True)
    )

    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(
            f"Ansatz inconsistente para k={k_value}: "
            f"esperados {expected}, encontrados {len(structure_df)}."
        )

    return decomposed, structure_df, tuple(sorted(initial_x_qubits))

def block_qubits_from_structure_row(row):
    i_value = int(row["i"])
    l_value = int(row["l"])
    gate_type = str(row["ansatz_gate_type"])
    if gate_type == "CY":
        return tuple(sorted((i_value, l_value)))
    if gate_type == "CCY":
        return tuple(sorted((i_value, i_value + 1, l_value)))
    raise ValueError(gate_type)

def physical_block_order(structure_local, n_value, k_value):
    pair_to_theta = {
        (int(r.l), int(r.i)): int(r.theta_index)
        for r in structure_local.itertuples(index=False)
    }
    order = []
    for l_value in range(int(n_value))[::-1]:
        for i_value in range(l_value - 1, l_value - 1 - int(k_value), -1):
            if i_value >= 0:
                key = (int(l_value), int(i_value))
                if key not in pair_to_theta:
                    raise RuntimeError(f"Bloco ausente para {key}")
                order.append(pair_to_theta[key])

    if sorted(order) != list(range(len(structure_local))):
        raise RuntimeError("A ordem física não cobre todos os parâmetros.")
    return order

_ANSATZ_CACHE = {}

def generic_ansatz_template(k_value):
    k_value = int(k_value)
    if k_value in _ANSATZ_CACHE:
        return _ANSATZ_CACHE[k_value]

    seed = RANDOM_SEED + 100 * N_ASSETS + k_value
    circuit, structure, initial_qubits = build_tracked_dicke_ansatz(
        N_ASSETS, k_value, seed
    )
    parameters = tuple(circuit.parameters)

    structure = structure.copy()
    structure["logical_block_qubits"] = structure.apply(
        block_qubits_from_structure_row, axis=1
    )
    order = physical_block_order(structure, N_ASSETS, k_value)
    pos = {theta_index: p for p, theta_index in enumerate(order)}
    structure["block_position"] = structure["theta_index"].map(pos)
    structure["block_position_norm"] = (
        structure["block_position"] / max(len(order) - 1, 1)
    )
    structure["n_qubits_touched"] = structure["logical_block_qubits"].map(len)

    # MESMA convenção usada na 20.20.
    structure["angular_period"] = structure["ansatz_gate_type"].map(
        {"CY": 4.0 * np.pi, "CCY": 2.0 * np.pi}
    )

    payload = {
        "k": k_value,
        "circuit": circuit,
        "parameters": parameters,
        "structure_df": structure.sort_values("theta_index").reset_index(drop=True),
        "initial_qubits": tuple(map(int, initial_qubits)),
        "physical_order": tuple(map(int, order)),
        "n_parameters": len(parameters),
    }
    _ANSATZ_CACHE[k_value] = payload
    return payload

audit = []
for k in (3, 4, 5):
    t = generic_ansatz_template(k)
    audit.append({
        "k": k,
        "n_parameters": t["n_parameters"],
        "expected": dicke_parameter_count(N_ASSETS, k),
        "passes": t["n_parameters"] == dicke_parameter_count(N_ASSETS, k),
    })

audit_df = pd.DataFrame(audit)
display(audit_df)
if not bool(audit_df["passes"].all()):
    raise RuntimeError("Falha na auditoria do ansatz.")



## Célula 5 — contexto clássico e medida quântica

A energia é calculada diretamente na base computacional. O bitstring ótimo **não entra na função objetivo**.


In [ ]:

# ============================================================
# CONTEXTO CLÁSSICO + MEDIDA
# ============================================================

def objective_for_instance(x_binary, mu_instance, sigma_instance):
    x = np.asarray(x_binary, dtype=float)
    return float(
        Q_VALUE * x @ sigma_instance @ x
        - (1.0 - Q_VALUE) * mu_instance @ x
        + RISK_FREE
    )

def exact_optimum_for_instance(k_value, mu_instance, sigma_instance):
    best_energy = np.inf
    best_bits = []

    for selected in combinations(range(N_ASSETS), int(k_value)):
        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected)] = 1
        energy = objective_for_instance(x, mu_instance, sigma_instance)
        bits_asset = "".join(map(str, x.tolist()))

        if energy < best_energy - ENERGY_ATOL:
            best_energy = float(energy)
            best_bits = [bits_asset]
        elif abs(energy - best_energy) <= ENERGY_ATOL:
            best_bits.append(bits_asset)

    return float(best_energy), tuple(sorted(best_bits))

def scenario_classical_context(k_value, asset_index, shock_sigma_units, risk_scale=1.0):
    mu_instance = np.asarray(mu, dtype=float).copy()
    sigma_instance = np.asarray(sigma, dtype=float).copy() * float(risk_scale)

    if int(asset_index) >= 0:
        mu_instance[int(asset_index)] += (
            float(shock_sigma_units) * float(mu_scale)
        )

    exact_energy, exact_bits_asset = exact_optimum_for_instance(
        int(k_value), mu_instance, sigma_instance
    )
    exact_bits_qiskit = tuple(sorted(bit[::-1] for bit in exact_bits_asset))

    labels = np.asarray(
        [format(i, f"0{N_ASSETS}b") for i in range(2 ** N_ASSETS)],
        dtype=object,
    )
    x_asset = np.asarray(
        [[int(ch) for ch in str(label)[::-1]] for label in labels],
        dtype=float,
    )

    basis_energies = (
        Q_VALUE * np.einsum("bi,ij,bj->b", x_asset, sigma_instance, x_asset)
        - (1.0 - Q_VALUE) * (x_asset @ mu_instance)
        + RISK_FREE
    )

    label_to_index = {str(label): int(i) for i, label in enumerate(labels)}
    optimal_indices = np.asarray(
        [label_to_index[bit] for bit in exact_bits_qiskit],
        dtype=int,
    )

    return {
        "k": int(k_value),
        "asset_index": int(asset_index),
        "shock_sigma_units": float(shock_sigma_units),
        "risk_scale": float(risk_scale),
        "mu": mu_instance,
        "sigma": sigma_instance,
        "exact_energy": exact_energy,
        "exact_bits_asset": exact_bits_asset,
        "exact_bits_qiskit": exact_bits_qiskit,
        "labels": labels,
        "label_to_index": label_to_index,
        "optimal_indices": optimal_indices,
        "basis_energies": np.asarray(basis_energies, dtype=float),
    }

def bind_state(template, theta_values):
    theta_values = np.asarray(theta_values, dtype=float)
    if theta_values.shape != (template["n_parameters"],):
        raise ValueError(
            f"theta shape={theta_values.shape}; "
            f"esperado {(template['n_parameters'],)}"
        )

    mapping = {
        template["parameters"][j]: float(theta_values[j])
        for j in range(template["n_parameters"])
    }
    bound = template["circuit"].assign_parameters(mapping, inplace=False)
    return np.asarray(Statevector.from_instruction(bound).data, dtype=np.complex128)

def measure_state(template, classical, theta_values, keep_state=True):
    state = bind_state(template, theta_values)
    probabilities = np.abs(state) ** 2
    dominant_index = int(np.argmax(probabilities))
    dominant_bitstring = str(classical["labels"][dominant_index])
    expected_energy = float(np.dot(probabilities, classical["basis_energies"]))

    result = {
        "expected_energy": expected_energy,
        "energy_gap": float(expected_energy - classical["exact_energy"]),
        "p_optimal": float(
            probabilities[classical["optimal_indices"]].sum()
        ),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probabilities[dominant_index]),
        "dominant_is_exact_optimum": bool(
            dominant_bitstring in set(classical["exact_bits_qiskit"])
        ),
    }

    if keep_state:
        result["statevector"] = state
    return result



## Célula 6 — QGT e rota dominante

São calculados com a mesma interpretação da 20.20:
- posto relativo do QGT;
- dimensão de participação;
- fração de parâmetros ativos para 95% do traço;
- rota do bitstring dominante ao longo da ordem física dos blocos.


In [ ]:

# ============================================================
# QGT
# ============================================================

def qgt_diagnostics(template, classical, theta_values):
    theta_values = np.asarray(theta_values, dtype=float)
    psi0 = bind_state(template, theta_values)

    derivatives = []
    for j in range(template["n_parameters"]):
        plus = theta_values.copy()
        minus = theta_values.copy()
        plus[j] += QGT_FD_STEP
        minus[j] -= QGT_FD_STEP
        derivatives.append(
            (bind_state(template, plus) - bind_state(template, minus))
            / (2.0 * QGT_FD_STEP)
        )

    derivatives = np.asarray(derivatives, dtype=np.complex128)
    gram = derivatives.conj() @ derivatives.T
    overlap = derivatives.conj() @ psi0
    metric = np.real(gram - np.outer(overlap, np.conj(overlap)))
    metric = 0.5 * (metric + metric.T)

    eig = np.clip(np.linalg.eigvalsh(metric), 0.0, None)[::-1]
    max_eig = float(eig[0]) if len(eig) else 0.0
    threshold = max(1e-14, QGT_REL_NULL_TOL * max_eig)
    numeric_rank = int(np.sum(eig > threshold))

    participation = (
        float(np.sum(eig) ** 2 / np.sum(eig ** 2))
        if np.sum(eig ** 2) > 0 else 0.0
    )

    diag = np.clip(np.diag(metric), 0.0, None)
    trace = float(diag.sum())

    if trace <= 1e-15:
        active_indices = tuple()
    else:
        order = np.argsort(diag)[::-1]
        cumulative = np.cumsum(diag[order]) / trace
        n_active = int(
            np.searchsorted(cumulative, ACTIVE_TRACE_COVERAGE, side="left") + 1
        )
        active_indices = tuple(sorted(map(int, order[:n_active])))

    return {
        "qgt_numeric_rank": numeric_rank,
        "qgt_rank_fraction": float(
            numeric_rank / max(template["n_parameters"], 1)
        ),
        "qgt_participation_dimension": participation,
        "qgt_trace": trace,
        "active_parameter_count": len(active_indices),
        "active_parameter_fraction": float(
            len(active_indices) / max(template["n_parameters"], 1)
        ),
    }

# ============================================================
# ROTA DOMINANTE
# ============================================================

def append_numeric_block(qc, row, theta_value):
    i_value = int(row["i"])
    l_value = int(row["l"])
    gate_type = str(row["ansatz_gate_type"])
    theta_value = float(theta_value)

    qc.cx(i_value, l_value)
    if gate_type == "CY":
        qc.cry(theta_value, l_value, i_value)
    elif gate_type == "CCY":
        qc.ry(theta_value, i_value)
        qc.ccx(l_value, i_value + 1, i_value)
        qc.ry(-theta_value, i_value)
        qc.ccx(l_value, i_value + 1, i_value)
    else:
        raise ValueError(gate_type)
    qc.cx(i_value, l_value)

def transition_signature(bit_a, bit_b):
    removed, added = [], []
    for position, (a, b) in enumerate(zip(str(bit_a), str(bit_b))):
        qubit = N_ASSETS - 1 - position
        if a == "1" and b == "0":
            removed.append(int(qubit))
        elif a == "0" and b == "1":
            added.append(int(qubit))
    hamming = int(sum(a != b for a, b in zip(str(bit_a), str(bit_b))))
    return {
        "removed_qubits": tuple(sorted(removed)),
        "added_qubits": tuple(sorted(added)),
        "hamming_distance": hamming,
        "n_replacements": int(hamming // 2),
    }

def dominant_route(template, classical, theta_values):
    structure = template["structure_df"].set_index("theta_index")

    qc = QuantumCircuit(N_ASSETS)
    for qubit in template["initial_qubits"]:
        qc.x(int(qubit))

    init_state = np.asarray(Statevector.from_instruction(qc).data, dtype=np.complex128)
    init_probs = np.abs(init_state) ** 2
    previous_bit = str(classical["labels"][int(np.argmax(init_probs))])
    initial_bit = previous_bit

    full_state = bind_state(template, theta_values)
    target_bit = max(
        classical["exact_bits_qiskit"],
        key=lambda bit: float(
            abs(full_state[classical["label_to_index"][bit]]) ** 2
        ),
    )

    rows = []
    route_step = 0

    for physical_step, theta_index in enumerate(template["physical_order"], start=1):
        row = structure.loc[int(theta_index)]
        append_numeric_block(qc, row, theta_values[int(theta_index)])

        state = np.asarray(Statevector.from_instruction(qc).data, dtype=np.complex128)
        probs = np.abs(state) ** 2
        current_bit = str(classical["labels"][int(np.argmax(probs))])

        if current_bit != previous_bit:
            route_step += 1
            sig = transition_signature(previous_bit, current_bit)
            rows.append({
                "route_step": route_step,
                "physical_block_step": physical_step,
                "theta_index_audit_only": int(theta_index),
                "from_bitstring": previous_bit,
                "to_bitstring": current_bit,
                **sig,
            })
            previous_bit = current_bit

    route_df = pd.DataFrame(rows)
    direct = transition_signature(initial_bit, target_bit)

    route_length = (
        int(route_df["n_replacements"].sum())
        if not route_df.empty else 0
    )
    minimum_length = int(direct["n_replacements"])
    stretch = (
        float(route_length / minimum_length)
        if minimum_length > 0 else 1.0
    )

    reconstructed = np.asarray(Statevector.from_instruction(qc).data, dtype=np.complex128)
    fidelity = float(abs(np.vdot(reconstructed, full_state)) ** 2)

    return route_df, {
        "initial_bitstring": initial_bit,
        "route_target_bitstring": target_bit,
        "endpoint_dominant_bitstring": previous_bit,
        "route_target_reached_as_dominant": bool(previous_bit == target_bit),
        "dominant_route_change_count": int(len(route_df)),
        "dominant_route_replacement_length": route_length,
        "minimum_replacement_length": minimum_length,
        "dominant_route_stretch": stretch,
        "route_reconstruction_fidelity": fidelity,
    }



## Célula 7 — carregar a varredura fina da 20.20

A tabela deve conter os 66 resultados:  
\(3\) cardinalidades \(\times 11\) pontos \(\times 2\) direções.

A identificação física abaixo **remove a direção do UID**. Isso corrige outro problema da primeira 20.21: forward e backward precisam ser pareados pelo mesmo \(H(s)\), não pelo `scenario_uid` direcional da 20.20.


In [ ]:

FINE_REQUIRED = {
    "direction",
    "k",
    "asset_index",
    "critical_return_shock_sigma_units",
    "delta_from_boundary",
    "return_shock_sigma_units",
}

def find_fine_scan():
    candidates = [
        ROOT / "vqe_r" / "pipeline_v20_17_compression_causal_geometry_pathsum"
        / "validation_v20_20_boundary_crossing" / "tables"
        / "fine_boundary_quantum_scan.csv",
        ROOT / "fine_boundary_quantum_scan.csv",
        Path.home() / "Downloads" / "fine_boundary_quantum_scan.csv",
    ]

    for p in candidates:
        if p.is_file():
            return p.resolve()

    for base in [ROOT, Path.home() / "Downloads"]:
        if not base.exists():
            continue
        for p in base.rglob("fine_boundary_quantum_scan.csv"):
            try:
                test = pd.read_csv(p, nrows=3)
                if FINE_REQUIRED.issubset(test.columns):
                    return p.resolve()
            except Exception:
                pass

    raise FileNotFoundError(
        "fine_boundary_quantum_scan.csv não encontrado. "
        "Execute a Célula 76 da 20.20 ou coloque o CSV no diretório do notebook."
    )

FINE_SCAN_CSV = find_fine_scan()
fine_df = pd.read_csv(FINE_SCAN_CSV)

missing = FINE_REQUIRED - set(fine_df.columns)
if missing:
    raise KeyError(f"Faltam colunas na varredura fina: {sorted(missing)}")

if "risk_scale" not in fine_df.columns:
    fine_df["risk_scale"] = 1.0

PHYSICAL_KEY = [
    "k",
    "asset_index",
    "return_shock_sigma_units",
    "risk_scale",
    "critical_return_shock_sigma_units",
    "delta_from_boundary",
]

def physical_uid_from_row(row):
    payload = "|".join(
        f"{c}={float(row[c]):.12g}" if c not in {"k", "asset_index"}
        else f"{c}={int(row[c])}"
        for c in PHYSICAL_KEY
    )
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

fine_df["physical_uid"] = fine_df.apply(physical_uid_from_row, axis=1)

physical_points_df = (
    fine_df
    .sort_values(["k", "delta_from_boundary", "direction"])
    .drop_duplicates("physical_uid")
    .reset_index(drop=True)
)

print("CSV =", FINE_SCAN_CSV)
print("linhas forward/backward =", len(fine_df))
print("pontos físicos únicos =", len(physical_points_df))

assert len(fine_df) >= len(physical_points_df)
assert fine_df["direction"].isin(["forward", "backward"]).all()

display(
    physical_points_df[
        ["physical_uid", "k", "asset_index",
         "critical_return_shock_sigma_units",
         "delta_from_boundary", "return_shock_sigma_units"]
    ]
)



## Célula 8 — auditoria antes da otimização

Antes de gastar 330 otimizações, esta célula verifica em cada \(k\):

1. dimensão do ansatz;
2. conservação de cardinalidade em um \(\theta\) aleatório;
3. reconstrução do ótimo clássico para o ponto \(\delta=0\);
4. consistência da energia exata com a tabela 20.20, quando disponível.


In [ ]:

audit_rows = []

for k_value in sorted(physical_points_df["k"].unique()):
    row = physical_points_df.query("k == @k_value").iloc[
        np.argmin(np.abs(physical_points_df.query("k == @k_value")["delta_from_boundary"].to_numpy()))
    ]

    template = generic_ansatz_template(int(k_value))
    classical = scenario_classical_context(
        int(row["k"]),
        int(row["asset_index"]),
        float(row["return_shock_sigma_units"]),
        float(row["risk_scale"]),
    )

    rng = np.random.default_rng(RANDOM_SEED + 21000 + int(k_value))
    periods = template["structure_df"]["angular_period"].to_numpy(dtype=float)
    theta_test = rng.uniform(-0.5, 0.5, size=template["n_parameters"]) * periods

    state = bind_state(template, theta_test)
    probs = np.abs(state) ** 2

    valid_mass = 0.0
    for idx, p in enumerate(probs):
        bit = format(idx, f"0{N_ASSETS}b")
        if bit.count("1") == int(k_value):
            valid_mass += float(p)

    exact_from_csv = np.nan
    if "exact_energy" in row.index and pd.notna(row["exact_energy"]):
        exact_from_csv = float(row["exact_energy"])

    audit_rows.append({
        "k": int(k_value),
        "n_parameters": int(template["n_parameters"]),
        "expected_n_parameters": dicke_parameter_count(N_ASSETS, int(k_value)),
        "cardinality_valid_mass": valid_mass,
        "exact_energy_recomputed": float(classical["exact_energy"]),
        "exact_energy_csv": exact_from_csv,
        "exact_energy_abs_difference": (
            abs(float(classical["exact_energy"]) - exact_from_csv)
            if pd.notna(exact_from_csv) else np.nan
        ),
    })

audit_20_21_df = pd.DataFrame(audit_rows)
audit_20_21_df["passes"] = (
    (audit_20_21_df["n_parameters"] == audit_20_21_df["expected_n_parameters"])
    & ((1.0 - audit_20_21_df["cardinality_valid_mass"]) < 1e-10)
    & (
        audit_20_21_df["exact_energy_abs_difference"].isna()
        | (audit_20_21_df["exact_energy_abs_difference"] < 1e-8)
    )
)

display(audit_20_21_df)

if not bool(audit_20_21_df["passes"].all()):
    raise RuntimeError(
        "Auditoria pré-campanha falhou. Não execute os cold starts antes de corrigir."
    )

audit_20_21_df.to_csv(TABLE_DIR / "pre_campaign_audit.csv", index=False)



## Célula 9 — solver cold-start

Para cada réplica:

\[
\theta_0^{(r)}
\sim
\mathrm{Uniforme}\!\left(-\frac{T_j}{2},+\frac{T_j}{2}\right),
\]

onde \(T_j\) é o período angular estrutural do bloco.

Depois, **uma única** otimização COBYLA é executada a partir daquele \(\theta_0^{(r)}\).  
Assim, `10 cold starts` significam realmente dez bacias iniciais independentes, e não dez chamadas que reutilizam o mesmo vetor ou o mesmo checkpoint.


In [ ]:

def cold_initial_theta(template, seed):
    rng = np.random.default_rng(int(seed))
    periods = (
        template["structure_df"]
        .sort_values("theta_index")["angular_period"]
        .to_numpy(dtype=float)
    )
    return rng.uniform(-0.5, 0.5, size=template["n_parameters"]) * periods

def optimize_one_cold_start(template, classical, seed, maxiter=COLD_MAXITER):
    theta0 = cold_initial_theta(template, seed)

    def objective(theta_values):
        state = bind_state(template, theta_values)
        probs = np.abs(state) ** 2
        return float(np.dot(probs, classical["basis_energies"]))

    initial_metrics = measure_state(
        template, classical, theta0, keep_state=False
    )

    result = minimize(
        objective,
        x0=np.asarray(theta0, dtype=float),
        method="COBYLA",
        options={
            "maxiter": int(maxiter),
            "rhobeg": 0.5,
            "catol": 1e-10,
        },
    )

    theta_opt = np.asarray(result.x, dtype=float)
    metrics = measure_state(template, classical, theta_opt, keep_state=True)

    return {
        "theta0": theta0,
        "theta_opt": theta_opt,
        "initial_expected_energy": float(initial_metrics["expected_energy"]),
        "initial_p_optimal": float(initial_metrics["p_optimal"]),
        "metrics": metrics,
        "nfev": int(getattr(result, "nfev", 0)),
        "optimizer_success_flag": bool(getattr(result, "success", False)),
        "optimizer_message": str(getattr(result, "message", "")),
    }

# Smoke test sem otimização longa.
test_row = physical_points_df.iloc[0]
test_template = generic_ansatz_template(int(test_row["k"]))
test_classical = scenario_classical_context(
    int(test_row["k"]),
    int(test_row["asset_index"]),
    float(test_row["return_shock_sigma_units"]),
    float(test_row["risk_scale"]),
)
theta0 = cold_initial_theta(test_template, RANDOM_SEED + 999)
test_measure = measure_state(test_template, test_classical, theta0, keep_state=False)

print("SMOKE TEST OK")
print("k =", test_row["k"])
print("theta0 shape =", theta0.shape)
print("P_opt inicial =", test_measure["p_optimal"])
print("energia inicial =", test_measure["expected_energy"])



## Célula 10 — executar 330 cold starts com checkpoint

O checkpoint é indexado por:

\[
(\texttt{physical\_uid},\ \texttt{cold\_replicate},\ \texttt{seed}),
\]

portanto **nenhuma seed reutiliza o resultado de outra**.


In [ ]:

COLD_CHECKPOINT = CHECKPOINT_DIR / "cold_runs_checkpoint.csv"

if COLD_CHECKPOINT.exists():
    cold_runs_df = pd.read_csv(COLD_CHECKPOINT)
    print("Checkpoint recuperado:", len(cold_runs_df), "linhas")
else:
    cold_runs_df = pd.DataFrame()

done = set()
if not cold_runs_df.empty:
    needed = {"physical_uid", "cold_replicate", "cold_seed"}
    if needed.issubset(cold_runs_df.columns):
        done = set(
            zip(
                cold_runs_df["physical_uid"].astype(str),
                cold_runs_df["cold_replicate"].astype(int),
                cold_runs_df["cold_seed"].astype(int),
            )
        )

records = [] if cold_runs_df.empty else cold_runs_df.to_dict("records")
total = len(physical_points_df) * N_COLD_STARTS

for point_index, row in physical_points_df.iterrows():
    uid = str(row["physical_uid"])
    k_value = int(row["k"])

    template = generic_ansatz_template(k_value)
    classical = scenario_classical_context(
        k_value,
        int(row["asset_index"]),
        float(row["return_shock_sigma_units"]),
        float(row["risk_scale"]),
    )

    for replicate in range(N_COLD_STARTS):
        cold_seed = int(
            RANDOM_SEED
            + 1_000_000
            + 10_000 * point_index
            + replicate
        )
        key = (uid, replicate, cold_seed)
        if key in done:
            continue

        current = len(done) + 1
        print(
            f"[20.21 cold] {current}/{total} "
            f"k={k_value} "
            f"delta={float(row['delta_from_boundary']):+.4f} "
            f"rep={replicate+1}/{N_COLD_STARTS} "
            f"seed={cold_seed}"
        )

        run = optimize_one_cold_start(
            template,
            classical,
            seed=cold_seed,
            maxiter=COLD_MAXITER,
        )

        qgt = qgt_diagnostics(
            template,
            classical,
            run["theta_opt"],
        )
        route_df, route = dominant_route(
            template,
            classical,
            run["theta_opt"],
        )

        m = run["metrics"]

        rec = {
            "physical_uid": uid,
            "cold_replicate": int(replicate),
            "cold_seed": int(cold_seed),
            "k": k_value,
            "k_over_n": float(k_value / N_ASSETS),
            "asset_index": int(row["asset_index"]),
            "critical_return_shock_sigma_units": float(
                row["critical_return_shock_sigma_units"]
            ),
            "delta_from_boundary": float(row["delta_from_boundary"]),
            "return_shock_sigma_units": float(row["return_shock_sigma_units"]),
            "risk_scale": float(row["risk_scale"]),
            "n_parameters": int(template["n_parameters"]),
            "exact_energy": float(classical["exact_energy"]),
            "exact_bitstrings_asset_order": tuple(classical["exact_bits_asset"]),
            "initial_expected_energy": float(run["initial_expected_energy"]),
            "initial_p_optimal": float(run["initial_p_optimal"]),
            "expected_energy": float(m["expected_energy"]),
            "energy_gap": float(m["energy_gap"]),
            "energy_gap_abs": abs(float(m["energy_gap"])),
            "p_optimal": float(m["p_optimal"]),
            "dominant_bitstring": str(m["dominant_bitstring"]),
            "dominant_probability": float(m["dominant_probability"]),
            "dominant_is_exact_optimum": bool(m["dominant_is_exact_optimum"]),
            **qgt,
            **route,
            "nfev": int(run["nfev"]),
            "optimizer_success_flag": bool(run["optimizer_success_flag"]),
        }

        rec["cold_success"] = bool(
            rec["p_optimal"] >= P_OPT_SUCCESS
            and rec["energy_gap_abs"] <= ENERGY_GAP_TOL
        )

        records.append(rec)
        done.add(key)

        # Salva após CADA réplica.
        cold_runs_df = pd.DataFrame(records)
        cold_runs_df.to_csv(COLD_CHECKPOINT, index=False)

cold_runs_df = pd.DataFrame(records)

print("\nCampanha concluída.")
print("shape =", cold_runs_df.shape)
display(cold_runs_df.head())



## Célula 11 — agregação por Hamiltoniano

Além da melhor solução, a dispersão entre cold starts é parte do resultado:
- fração de sucesso;
- mediana e máximo de \(P_{\rm opt}\);
- número de bacias dominantes;
- entropia dos estados dominantes;
- QGT e rota.


In [ ]:

def entropy_bits(values):
    p = (
        pd.Series(values)
        .dropna()
        .astype(str)
        .value_counts(normalize=True)
    )
    if len(p) == 0:
        return np.nan
    return float(-(p * np.log2(p)).sum())

cold_summary_df = (
    cold_runs_df
    .groupby("physical_uid", as_index=False)
    .agg(
        k=("k", "first"),
        asset_index=("asset_index", "first"),
        critical_return_shock_sigma_units=(
            "critical_return_shock_sigma_units", "first"
        ),
        delta_from_boundary=("delta_from_boundary", "first"),
        return_shock_sigma_units=("return_shock_sigma_units", "first"),
        risk_scale=("risk_scale", "first"),
        exact_energy=("exact_energy", "first"),
        n_parameters=("n_parameters", "first"),
        n_cold=("cold_seed", "count"),
        cold_success_fraction=("cold_success", "mean"),
        cold_p_optimal_best=("p_optimal", "max"),
        cold_p_optimal_median=("p_optimal", "median"),
        cold_p_optimal_min=("p_optimal", "min"),
        cold_energy_gap_best=("energy_gap_abs", "min"),
        cold_energy_gap_median=("energy_gap_abs", "median"),
        cold_unique_dominant_states=("dominant_bitstring", "nunique"),
        cold_state_entropy_bits=("dominant_bitstring", entropy_bits),
        qgt_rank_fraction_median=("qgt_rank_fraction", "median"),
        qgt_rank_fraction_std=("qgt_rank_fraction", "std"),
        qgt_participation_median=("qgt_participation_dimension", "median"),
        active_parameter_fraction_median=("active_parameter_fraction", "median"),
        route_stretch_median=("dominant_route_stretch", "median"),
        route_target_fraction=(
            "route_target_reached_as_dominant", "mean"
        ),
        nfev_median=("nfev", "median"),
    )
)

cold_summary_df.to_csv(
    TABLE_DIR / "cold_start_summary_20_21.csv",
    index=False,
)

display(cold_summary_df)



## Célula 12 — parear cold, forward e backward pelo mesmo Hamiltoniano

A comparação agora é feita pelo `physical_uid`, que não contém a direção da varredura.


In [ ]:

CONT_METRICS = [
    c for c in [
        "p_optimal",
        "expected_energy",
        "dominant_bitstring",
        "qgt_numeric_rank",
        "qgt_rank_fraction",
        "qgt_participation_dimension",
        "active_parameter_fraction",
        "dominant_route_replacement_length",
        "dominant_route_stretch",
        "route_target_reached_as_dominant",
        "nfev_total",
    ]
    if c in fine_df.columns
]

cont = (
    fine_df[
        ["physical_uid", "direction"] + CONT_METRICS
    ]
    .drop_duplicates(["physical_uid", "direction"])
)

wide = cont.pivot(
    index="physical_uid",
    columns="direction",
    values=CONT_METRICS,
)

wide.columns = [
    f"{metric}_{direction}"
    for metric, direction in wide.columns
]
wide = wide.reset_index()

comparison_df = cold_summary_df.merge(
    wide,
    on="physical_uid",
    how="left",
)

# Auditoria de pareamento.
pair_audit = pd.DataFrame({
    "n_physical_points": [len(comparison_df)],
    "n_with_forward": [
        comparison_df.filter(regex="_forward$").notna().any(axis=1).sum()
        if len(comparison_df.filter(regex="_forward$").columns) else 0
    ],
    "n_with_backward": [
        comparison_df.filter(regex="_backward$").notna().any(axis=1).sum()
        if len(comparison_df.filter(regex="_backward$").columns) else 0
    ],
})

display(pair_audit)
display(comparison_df.head(20))

comparison_df.to_csv(
    TABLE_DIR / "cold_vs_continuation_raw_20_21.csv",
    index=False,
)



## Célula 13 — classificação causal

A regra é conservadora:

- **robust**: cold, forward e backward recuperam o mesmo regime;
- **path-dependent / multimodal**: forward e backward discordam e cold starts ocupam mais de uma bacia;
- **warm-start trapping**: continuação falha, mas cold starts recuperam o ótimo de forma robusta;
- **continuation advantage**: continuação recupera o ótimo, cold starts frequentemente não;
- **persistent optimization failure**: nenhum protocolo recupera de modo confiável.

A última classe **não é automaticamente “limitação do ansatz”**.


In [ ]:

def colval(row, name):
    return row[name] if name in row.index else np.nan

def p_success(p):
    return pd.notna(p) and float(p) >= P_OPT_SUCCESS

def classify_causal_regime(row):
    cold_frac = float(row["cold_success_fraction"])

    p_fw = colval(row, "p_optimal_forward")
    p_bw = colval(row, "p_optimal_backward")

    fw_ok = p_success(p_fw)
    bw_ok = p_success(p_bw)

    dom_fw = colval(row, "dominant_bitstring_forward")
    dom_bw = colval(row, "dominant_bitstring_backward")
    dominant_disagreement = (
        pd.notna(dom_fw)
        and pd.notna(dom_bw)
        and str(dom_fw) != str(dom_bw)
    )

    many_cold_basins = (
        int(row["cold_unique_dominant_states"]) >= 2
    )

    if cold_frac >= 0.8 and fw_ok and bw_ok and not dominant_disagreement:
        return "robust"

    if dominant_disagreement and many_cold_basins:
        return "path_dependent_multimodal"

    if cold_frac >= 0.8 and (not fw_ok or not bw_ok):
        return "warm_start_trapping_candidate"

    if cold_frac <= 0.2 and (fw_ok or bw_ok):
        return "continuation_advantage"

    if cold_frac <= 0.2 and not fw_ok and not bw_ok:
        return "persistent_optimization_failure"

    return "mixed_regime"

comparison_df["causal_regime"] = comparison_df.apply(
    classify_causal_regime,
    axis=1,
)

regime_counts_df = (
    comparison_df["causal_regime"]
    .value_counts()
    .rename_axis("causal_regime")
    .reset_index(name="count")
)

display(regime_counts_df)

comparison_df.to_csv(
    TABLE_DIR / "cold_vs_continuation_classification_20_21.csv",
    index=False,
)
regime_counts_df.to_csv(
    TABLE_DIR / "causal_regime_counts_20_21.csv",
    index=False,
)



## Célula 14 — teste forte de aprisionamento por warm start

O caso mais limpo é:

\[
P_{\rm opt}^{\rm continuation}<0.1,
\qquad
\max_r P_{\rm opt}^{\rm cold}(r)\ge0.9.
\]

Isso demonstra que o mesmo ansatz e o mesmo Hamiltoniano admitem uma boa solução a partir de outra bacia inicial.


In [ ]:

def continuation_best(row):
    vals = []
    for c in ["p_optimal_forward", "p_optimal_backward"]:
        if c in row.index and pd.notna(row[c]):
            vals.append(float(row[c]))
    return max(vals) if vals else np.nan

comparison_df["continuation_best_p_optimal"] = comparison_df.apply(
    continuation_best,
    axis=1,
)

comparison_df["strong_warm_start_trap"] = (
    (comparison_df["continuation_best_p_optimal"] < 0.10)
    & (comparison_df["cold_p_optimal_best"] >= P_OPT_SUCCESS)
)

strong_traps_df = comparison_df.loc[
    comparison_df["strong_warm_start_trap"],
    [
        "physical_uid",
        "k",
        "delta_from_boundary",
        "asset_index",
        "continuation_best_p_optimal",
        "cold_p_optimal_best",
        "cold_success_fraction",
        "cold_unique_dominant_states",
        "cold_state_entropy_bits",
    ],
].copy()

display(strong_traps_df)

strong_traps_df.to_csv(
    TABLE_DIR / "strong_warm_start_traps_20_21.csv",
    index=False,
)



## Célula 15 — figuras separadas por pergunta

Não há score composto. Cada figura testa uma hipótese diferente.


In [ ]:

for k_value, g in comparison_df.groupby("k"):
    g = g.sort_values("delta_from_boundary")

    # A. cold success
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.plot(
        g["delta_from_boundary"],
        g["cold_success_fraction"],
        marker="o",
    )
    ax.axvline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xlabel(r"$\delta=s-s_c$")
    ax.set_ylabel("fração de cold starts que recupera o ótimo")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"20.21 — robustez a cold start, k={int(k_value)}")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"cold_success_fraction_k{int(k_value)}.png",
        dpi=180,
    )
    plt.show()

    # B. comparação Popt
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.plot(
        g["delta_from_boundary"],
        g["cold_p_optimal_median"],
        marker="o",
        label="cold: mediana",
    )

    if "p_optimal_forward" in g.columns:
        ax.plot(
            g["delta_from_boundary"],
            g["p_optimal_forward"],
            marker="o",
            label="forward",
        )

    if "p_optimal_backward" in g.columns:
        ax.plot(
            g["delta_from_boundary"],
            g["p_optimal_backward"],
            marker="o",
            label="backward",
        )

    ax.axvline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xlabel(r"$\delta=s-s_c$")
    ax.set_ylabel(r"$P_{\rm opt}$")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"20.21 — continuação vs cold start, k={int(k_value)}")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"continuation_vs_cold_poptimal_k{int(k_value)}.png",
        dpi=180,
    )
    plt.show()

    # C. diversidade de bacias
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.plot(
        g["delta_from_boundary"],
        g["cold_unique_dominant_states"],
        marker="o",
    )
    ax.axvline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xlabel(r"$\delta=s-s_c$")
    ax.set_ylabel("estados dominantes distintos / 10 cold starts")
    ax.set_title(f"20.21 — multiplicidade de bacias, k={int(k_value)}")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"cold_basin_diversity_k{int(k_value)}.png",
        dpi=180,
    )
    plt.show()

    # D. QGT cold
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.plot(
        g["delta_from_boundary"],
        g["qgt_rank_fraction_median"],
        marker="o",
    )
    ax.axvline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xlabel(r"$\delta=s-s_c$")
    ax.set_ylabel(r"mediana $\mathrm{rank}(G)/N_\theta$")
    ax.set_title(f"20.21 — geometria sob cold starts, k={int(k_value)}")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"cold_qgt_rank_fraction_k{int(k_value)}.png",
        dpi=180,
    )
    plt.show()



## Célula 16 — manifesto científico

O ponto central desta 20.21 é separar três objetos:

\[
\boxed{
\text{mudança clássica do ótimo}
\neq
\text{memória do otimizador}
\neq
\text{limitação do ansatz}
}
\]

Cold starts resolvem principalmente o segundo termo.  
Eles não bastam, sozinhos, para provar o terceiro.


In [ ]:

manifest = {
    "experiment": "20.21 cold-start vs continuation v2 autonomous",
    "source_merge_pkl": str(MERGE_PKL),
    "source_fine_scan_csv": str(FINE_SCAN_CSV),
    "n_assets": int(N_ASSETS),
    "k_values": sorted(map(int, physical_points_df["k"].unique())),
    "n_physical_points": int(len(physical_points_df)),
    "n_cold_starts_per_point": int(N_COLD_STARTS),
    "planned_cold_optimizations": int(
        len(physical_points_df) * N_COLD_STARTS
    ),
    "optimizer": "COBYLA",
    "maxiter_per_cold_start": int(COLD_MAXITER),
    "cold_initialization": (
        "independent uniform theta_j in [-T_j/2, +T_j/2]"
    ),
    "label_leakage_rule": (
        "exact optimum is evaluation label only; never optimizer input"
    ),
    "interpretation": {
        "persistent_failure": (
            "not automatically interpreted as ansatz limitation"
        ),
        "classical_boundary": (
            "delta=0 is classical/combinatorial evidence, not quantum evidence"
        ),
    },
    "regime_counts": (
        regime_counts_df.set_index("causal_regime")["count"]
        .astype(int)
        .to_dict()
    ),
}

with open(
    TABLE_DIR / "experiment_20_21_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(json.dumps(manifest, indent=2, ensure_ascii=False))

print("\nArquivos finais:")
for p in [
    TABLE_DIR / "pre_campaign_audit.csv",
    TABLE_DIR / "cold_start_summary_20_21.csv",
    TABLE_DIR / "cold_vs_continuation_raw_20_21.csv",
    TABLE_DIR / "cold_vs_continuation_classification_20_21.csv",
    TABLE_DIR / "strong_warm_start_traps_20_21.csv",
    TABLE_DIR / "experiment_20_21_manifest.json",
]:
    print(" -", p)
